# CV08 — Flow matching từ noise đến ảnh

Nối noise z và ảnh x bằng x_t=(1-t)z+t*x. Target là vận tốc x-z; sampler bắt đầu t=0 rồi cộng vận tốc theo dt dương tới t=1.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Nối noise z và ảnh x bằng x_t=(1-t)z+t*x. Target là vận tốc x-z; sampler bắt đầu t=0 rồi cộng vận tốc theo dt dương tới t=1. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv08_flow import Velocity, loss_fn, sample
images = torch.rand(2, 3, 32, 32, device=device) * 2 - 1
model = Velocity(d=32).to(device)
batch = (images,)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
noise = torch.randn_like(images)
times = torch.tensor([0.25, 0.75], device=device)
interpolated = (1 - times[:, None, None, None]) * noise + times[:, None, None, None] * images
velocity = model(interpolated, times)
print('Velocity field:', velocity.shape)
print('Target velocity:', (images - noise).shape)


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

CNN vận tốc → time-conditioned U-Net → conditional flow restoration/solver tốt hơn. Khôi phục ảnh bằng flow cần nhánh điều kiện; ví dụ này chưa có.


In [ ]:
model.eval()
with torch.no_grad():
    generated = sample(model, torch.randn_like(images), steps=8)
print('Euler samples:', generated.shape)
print('Finite samples:', torch.isfinite(generated).all().item())
print('One gradient step cannot generate meaningful images; this checks the ODE path only.')


## Thử nghiệm tiếp theo

CNN vận tốc → time-conditioned U-Net → conditional flow restoration/solver tốt hơn. Khôi phục ảnh bằng flow cần nhánh điều kiện; ví dụ này chưa có.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv08_flow.py) và [tài liệu gốc](../REFERENCE.md).
